# 01 EDA — OpenI IU Chest X-ray
Validate label frequencies, image stats, and patient uniqueness across splits.


In [ ]:
import pandas as pd
import os

studies = pd.read_csv('../data/processed/studies.csv')
print(f'Total studies: {len(studies)}')
print(f'Columns: {list(studies.columns)}')
print()
print(studies.head(3).to_string())


## 1. Text Field Coverage


In [ ]:
n = len(studies)
missing_findings = (studies['findings'] == '').sum()
missing_impression = (studies['impression'] == '').sum()
missing_report = (studies['report_text'] == '').sum()

print(f'Missing FINDINGS:   {missing_findings} / {n} ({100*missing_findings/n:.1f}%)')
print(f'Missing IMPRESSION: {missing_impression} / {n} ({100*missing_impression/n:.1f}%)')
print(f'Empty report_text:  {missing_report} / {n} ({100*missing_report/n:.1f}%)')


## 2. Word Count Distribution


In [ ]:
studies['findings_words'] = studies['findings'].str.lower().str.split().apply(lambda x: len(x) if isinstance(x, list) else 0)
studies['impression_words'] = studies['impression'].str.lower().str.split().apply(lambda x: len(x) if isinstance(x, list) else 0)
print('FINDINGS word count stats:')
print(studies['findings_words'].describe())
print()
print('IMPRESSION word count stats:')
print(studies['impression_words'].describe())


## 3. Common Keyword Frequencies (Rough Label Proxy)


In [ ]:
keywords = ['cardiomegaly', 'effusion', 'edema', 'pneumothorax', 'consolidation', 'no acute', 'normal']
combined = (studies['findings'] + ' ' + studies['impression']).str.lower()

print('Keyword frequencies in findings+impression:')
for kw in keywords:
    count = combined.str.contains(kw, na=False).sum()
    print(f'  {kw:<20} {count:>5} / {n} ({100*count/n:.1f}%)')


## 4. Split Leakage Check


In [ ]:
train = pd.read_csv('../data/splits/train.csv')
val = pd.read_csv('../data/splits/val.csv')
test = pd.read_csv('../data/splits/test.csv')
cal = pd.read_csv('../data/splits/calibration.csv')

train_ids = set(train['study_id'])
val_ids = set(val['study_id'])
test_ids = set(test['study_id'])
cal_ids = set(cal['study_id'])

print(f'train: {len(train_ids)}, val: {len(val_ids)}, test: {len(test_ids)}, cal: {len(cal_ids)}')
print(f'train ∩ val:  {len(train_ids & val_ids)}')
print(f'train ∩ test: {len(train_ids & test_ids)}')
print(f'val ∩ test:   {len(val_ids & test_ids)}')

assert len(train_ids & val_ids) == 0
assert len(train_ids & test_ids) == 0
assert len(val_ids & test_ids) == 0
assert len(cal_ids) > 0
print('\nPatient uniqueness check PASSED.')
